In [9]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
import pandas as pd
import numpy as np

In [10]:
# Evaluation metrics used in the paper:

def paper_r2(y_true, y_pred):
    """Paper's metric: Pearson correlation"""
    corr = np.corrcoef(y_true, y_pred)[0, 1]
    return corr 

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred)**2))

In [13]:
# Definition of attributes and targets

selected_attributes = [
    'np_without_modification', 'surface_modification', 'size_tem', 'zeta_potential', 
    'incubation_protein_source', 'incubation_plasma_concentration', 'incubation_np_concentration',
    'centrifugation_speed', 'centrifugation_time', 'centrifugation_temperature', 'np_type',
    'np_shape', 'dispersion_medium', 'dispersion_medium_ph', 'size_dls', 'pdi', 'incubation_culture',
    'incubation_time', 'incubation_temperature', 'centrifugation_repetitions', 'modification_type'
]

categorical_attributes = [
    'np_without_modification', 'surface_modification', 'incubation_protein_source',
    'np_type', 'np_shape', 'dispersion_medium', 'incubation_culture', 'modification_type'
]

protein_targets = [
    "P01871", "P01024", "P02647", "P02649", "P02768", "P04004", "P01834", 
    "P10909", "P02652", "P00734", "P01009", "P01042", "P01857", "P01859", 
    "P01023", "P01619", "P04196", "P02656", "P04114", "P02787", "P02766", 
    "P06396", "P06727", "P0C0L5", "P02671", "P08603", "P02765", "Q14624", 
    "P0DOY2", "P01008", "P68871", "P01764", "P04003", "P0C0L4", "P01876", 
    "P02749", "P02675", "P01860", "P01011", "P00747", "P02774", "P00751", 
    "P19823", "P08697", "P19827", "P69905", "P02760", "P02748", "P02679", 
    "P02790", "B9A064", "P07996", "P05155", "P01766", "P12259", "P00738",
    "P02751", "P02654", "P04406", "P00739", "P02655", "P00736", "P07225", 
    "P05154", "P09871", "P60709", "P03952", "P02747", "P05546", "P02746",
    "P27169", "P01019", "P35542", "P04217", "Q14520", "P02743", "P02763", 
    "P49908", "O43866", "P18428", "P55056", "P04264", "P00748", "P01591", 
    "P01861", "Q92954", "P02776", "P0DJI8", "P01031", "P05156", "P13671", 
    "Q03591", "P06312", "P00740", "P01615", "P00742", "P04070", "O14791", 
    "P05090", "P02745", "P00488", "Q13103", "P01877", "P01700", "P10643", 
    "Q9UK55", "P03951", "Q06033", "Q96IY4", "P13645", "P04433", "P07358",
    "P27918", "P05452", "P20851", "P07357", "P07360", "P01599", "O95445", 
    "Q9BXR6", "P02775", "P02741", "P35579", "P15169", "Q13790", "P35858", 
    "P49747", "P36955", "P19652", "P07737", "P22891", "P35908", "P80748", 
    "P08514", "P01593", "Q04756", "P06733", "P23528", "P63104", "P18065", 
    "P08519", "Q86UX7", "P02753", "Q5TB80", "P22352", "P00746", "P35443", 
    "P62937", "P10720", "P48740", "P25311", "P43652", "P35527", "Q9Y490", 
    "P05106", "P17936", "P18206", "P02788", "P06702", "P01701", "Q6Q788", 
    "Q96PD5", "Q9UGM5", "O00391", "P11142", "P14618", "P23142", "P68366",
    "P12814", "P61224", "Q13201", "P67936", "P0DOY3", "P08709", "P01034", 
    "P81605", "P02533", "P11226"]

In [14]:
# Load dataset: 
df = pd.read_csv('individual_proteins_dataset.csv')

# Get selected attributes from dataset:
X = df[selected_attributes]

# One-hot encode categorical attributes:
X = pd.get_dummies(X, columns=categorical_attributes)

kf = KFold(n_splits=10, shuffle=True, random_state=42)

all_target_r2_means = []
all_target_rmse_means = []

results = [] 

for target in protein_targets:
    y = df[target].values

    fold_r2_scores = []
    fold_rmse_scores = []

    print(f"\n==================== Target {target} ====================")

    for fold_idx, (train_idx, test_idx) in enumerate(kf.split(X), start=1):
        
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        # Modifique: random_state de 42 a 1234 (como se muestra en el repositorio del paper)
        # Ademas modifique: max_samples=0.3 (lo que supongo que es n_samples)
        
        model = RandomForestRegressor(n_estimators=500, random_state=42, max_samples=0.3) 
        model.fit(X_train, y_train)

        preds = model.predict(X_test)

        # Compute R2 and RMSE metrics for this fold:
       
        r2_score = paper_r2(y_test, preds)
        rmse_score = rmse(y_test, preds)
       
        fold_r2_scores.append(r2_score)
        fold_rmse_scores.append(rmse_score)

        print(f"Fold {fold_idx}: R2={r2_score:.4f}, RMSE={rmse_score:.4f}")

        # Save this fold's result: 
        results.append({
            "target": target,
            "fold": fold_idx,
            "r2": r2_score,
            "rmse": rmse_score
        })

    # Compute mean metrics for this protein target:
    mean_r2 = np.mean(fold_r2_scores)
    mean_rmse = np.mean(fold_rmse_scores)

    all_target_r2_means.append(mean_r2)
    all_target_rmse_means.append(mean_rmse)

    print(f"Mean R2 for {target}:  {mean_r2:.4f}")
    print(f"Mean RMSE for {target}: {mean_rmse:.4f}")

    # Save the mean result as an extra row
    results.append({
        "target": target,
        "fold": "mean",
        "r2": mean_r2,
        "rmse": mean_rmse
    })


==================== Target P01871 ====================
Fold 1: R2=0.6922, RMSE=0.8180
Fold 2: R2=0.6278, RMSE=1.0267
Fold 3: R2=0.7164, RMSE=0.8713
Fold 4: R2=0.7448, RMSE=0.6844
Fold 5: R2=0.7762, RMSE=1.0424
Fold 6: R2=0.6623, RMSE=1.0228
Fold 7: R2=0.6461, RMSE=0.7150
Fold 8: R2=0.6246, RMSE=0.8587
Fold 9: R2=0.6100, RMSE=0.7517
Fold 10: R2=0.5653, RMSE=0.7618
Mean R2 for P01871:  0.6666
Mean RMSE for P01871: 0.8553

==================== Target P01024 ====================
Fold 1: R2=0.4820, RMSE=3.1069
Fold 2: R2=0.7305, RMSE=1.0600
Fold 3: R2=0.8075, RMSE=0.8580
Fold 4: R2=0.7404, RMSE=1.1437
Fold 5: R2=0.7149, RMSE=0.7691
Fold 6: R2=0.6887, RMSE=0.7749
Fold 7: R2=0.2718, RMSE=1.3024
Fold 8: R2=0.6852, RMSE=0.8287
Fold 9: R2=0.7078, RMSE=0.9433
Fold 10: R2=0.5929, RMSE=0.9446
Mean R2 for P01024:  0.6422
Mean RMSE for P01024: 1.1732

==================== Target P02647 ====================
Fold 1: R2=0.5332, RMSE=2.5962
Fold 2: R2=0.6409, RMSE=3.6471
Fold 3: R2=0.7033, RMSE=3.1959


In [15]:
# Compute final averages across all targets:
final_r2_average = np.mean(all_target_r2_means)
final_rmse_average = np.mean(all_target_rmse_means)

print("\n==================== Final Results ====================")
print(f"Final average R2:               {final_r2_average:.4f}")
print(f"Final average RMSE:             {final_rmse_average:.4f}")

df_results = pd.DataFrame(results)
df_results.to_csv("cross_validation_results_42_24a.csv", index=False)

print("\n==================== Saved ====================")
print("Results written to cross_validation_results.csv")


==================== Final Results ====================
Final average R2:               0.7190
Final average RMSE:             0.6214

==================== Saved ====================
Results written to cross_validation_results.csv
